# Data Exploration - Premier League 2024

**Date**: November 1, 2025  
**Goal**: Explore PL event data and validate data loading pipeline

## Steps:
1. ✅ Load event data and metadata
2. ⏳ Coordinate normalization and attack direction
3. ⏳ Pass classification (9 types)
4. ⏳ Data quality assessment
5. ⏳ Grid validation
6. ⏳ Visualization

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import (
    load_premier_league_events,
    load_match_metadata,
    extract_pass_events,
    rescale_coordinates,
    get_data_summary
)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## Step 1: Load Sample Data

In [ ]:
# Load a small sample for exploration
print("Loading sample data (10 matches)...")
events = load_premier_league_events(limit_matches=10)
metadata = load_match_metadata()

In [ ]:
# Get comprehensive summary
get_data_summary(events)

In [ ]:
# Check metadata
print(f"Metadata loaded for {len(metadata)} matches")
metadata.head(10)

## Step 2: Extract and Explore Passes

In [ ]:
# Extract pass events
passes = extract_pass_events(events)

# Rescale coordinates
passes = rescale_coordinates(passes)

print(f"\nExtracted {len(passes):,} passes")
print(f"Success rate: {passes['success'].mean():.1%}")
passes.head()

## Step 3: Check Coordinate Ranges

In [ ]:
# Original coordinates (SkillCorner)
print("Original Coordinates (SkillCorner):")
print(f"x_start: [{passes['x_start'].min():.2f}, {passes['x_start'].max():.2f}]")
print(f"y_start: [{passes['y_start'].min():.2f}, {passes['y_start'].max():.2f}]")

# Rescaled coordinates (FIFA)
print("\nRescaled Coordinates (FIFA 105×68m):")
print(f"x_start_rescaled: [{passes['x_start_rescaled'].min():.2f}, {passes['x_start_rescaled'].max():.2f}]")
print(f"y_start_rescaled: [{passes['y_start_rescaled'].min():.2f}, {passes['y_start_rescaled'].max():.2f}]")

## Step 4: Check Attack Direction by Period

In [ ]:
# Check if coordinates are normalized for attack direction
print("Attack Direction Check (avg x position by team and period):")
print("="*70)

for match_id in passes['match_id'].unique()[:3]:
    match_passes = passes[passes['match_id'] == match_id]
    print(f"\nMatch {match_id}:")
    
    for team_id in match_passes['team_id'].unique():
        team_name = match_passes[match_passes['team_id'] == team_id]['team_shortname'].iloc[0]
        print(f"  Team {team_name}:")
        
        for period in [1, 2]:
            subset = match_passes[
                (match_passes['team_id'] == team_id) & 
                (match_passes['period'] == period)
            ]
            if len(subset) > 0:
                avg_x = subset['x_start'].mean()
                print(f"    Period {period}: avg x = {avg_x:7.2f} (n={len(subset):3d} passes)")

## Observations

**Coordinate System**: SkillCorner uses center origin (-52 to +52, -34 to +34)  
**Attack Direction**: ❌ NOT normalized - teams switch sides at halftime  
**Next Step**: Implement attack direction normalization (Step 2 tomorrow)

## Step 5: Explore Available Columns

In [ ]:
# Check what useful columns we have
print("Useful columns for MDP:")
useful_cols = [
    'match_id', 'period', 'team_id', 'team_shortname',
    'x_start', 'y_start', 'x_end', 'y_end',
    'x_start_rescaled', 'y_start_rescaled', 'x_end_rescaled', 'y_end_rescaled',
    'end_type', 'pass_distance', 'pass_direction',
    'pass_outcome', 'success'
]

available = [col for col in useful_cols if col in passes.columns]
print(f"Available: {available}")

passes[available].head(10)

## Summary

✅ **Completed**:
- Data loading function (`load_premier_league_events`)
- Metadata loading (`load_match_metadata`)
- Pass extraction (`extract_pass_events`)
- Coordinate rescaling (`rescale_coordinates`)
- Data summary function (`get_data_summary`)

📋 **Next Steps** (Nov 2):
1. Implement attack direction normalization
2. Implement pass classification (9 types)
3. Validate classifications with visualizations